# SHIPIT Agent: Streaming Events And Packets

Use this notebook to inspect the real-time runtime stream, collected event packets, and frontend-ready packet transports.

In [1]:
from pathlib import Path
import sys
import json
import textwrap
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.run_multi_tool_agent import build_demo_agent, build_llm_from_env

In [2]:
def _short(val, n=220):
    s = val if isinstance(val, str) else json.dumps(val, default=str)
    return textwrap.shorten(s.replace('\n', ' '), width=n, placeholder=' …')

def pretty_event(ev):
    t, msg, p = ev.type, ev.message or '', ev.payload or {}
    if t == 'run_started':
        return f'[status] run started :: {_short(p.get("prompt", ""), 140)}'
    if t == 'planning_started':
        return f'[thinking] planning started :: {_short(p.get("prompt", ""), 140)}'
    if t == 'planning_completed':
        return f'[thinking] planning completed :: {_short(p.get("output", ""))}'
    if t == 'reasoning_started':
        return f'[thinking] reasoning started :: {_short(p)}'
    if t == 'reasoning_completed':
        return f'[thinking] reasoning completed :: {_short(p)}'
    if t == 'tool_called':
        return f'[tool] calling {msg} :: args={_short(p.get("arguments", {}))}'
    if t == 'tool_completed':
        return f'[tool] completed {msg} :: {_short(p.get("output", ""))}'
    if t == 'interactive_request':
        return f'[interactive] {msg} :: {_short(p)}'
    if t == 'run_completed':
        return f'[final] {_short(p.get("output", ""))}'
    return f'[{t}] {msg} :: {_short(p)}'


In [3]:
llm = build_llm_from_env('bedrock')

agent = build_demo_agent(
    llm=llm,
    workspace_root=str(ROOT / '.shipit_notebooks' / 'streaming_demo'),
)

prompt = 'Use tools if useful and explain the runtime steps.'

In [5]:
# Live streaming status view with incremental notebook updates.
# Note: the first visible update still depends on when the runtime yields its first event.
lines = []
handle = display(Markdown('## Live Stream\n\nWaiting for first event...'), display_id=True)

for index, event in enumerate(agent.stream(prompt), start=1):
    lines.append(f'{index}. `{event.type}` {pretty_event(event)}')
    handle.update(Markdown('## Live Stream\n\n' + '\n'.join(lines)))

lines

## Live Stream

1. `run_started` [status] run started :: Use tools if useful and explain the runtime steps.
2. `step_started` [step_started] LLM completion started :: {"tool_count": 28, "iteration": 1}
3. `tool_called` [tool] calling Tool called: ask_user :: args={"question": "What specific information would you like me to retrieve?", "options": [{"label": "Current Bitcoin price (USD)"}, {"label": "Current price of another cryptocurrency (e.g., Ethereum)"}, {"label": "Weather …
4. `tool_completed` [tool] completed Tool completed: ask_user :: {"kind": "ask_user", "question": "What specific information would you like me to retrieve?", "context": "", "options": [{"label": "Current Bitcoin price (USD)"}, {"label": "Current price of another cryptocurrency …
5. `interactive_request` [interactive] Interactive request from ask_user :: {"kind": "ask_user", "payload": {"interactive": true, "kind": "ask_user", "question": "What specific information would you like me to retrieve?", "context": "", "options": [{"label": "Current Bitcoin price (USD)"}, …
6. `step_started` [step_started] LLM completion started :: {"tool_count": 28, "iteration": 2}
7. `tool_called` [tool] calling Tool called: web_search :: args={"max_results": 5, "query": "CoinGecko Bitcoin price USD"}
8. `tool_completed` [tool] completed Tool completed: web_search :: [1] BTC to USD: Bitcoin Price in US Dollar | CoinGecko Calculate Bitcoin's price in US Dollar by inputting your desired amount on CoinGecko's BTC to USD converter. Track its historical price movements on the BTC to USD …
9. `step_started` [step_started] LLM completion started :: {"tool_count": 28, "iteration": 3}
10. `run_completed` [final] Below is a **generic “run‑through”** that shows how I would use the available tools to answer a fresh‑information request, with every runtime step made explicit. I’ll illustrate the flow with a concrete example (the …

['1. `run_started` [status] run started :: Use tools if useful and explain the runtime steps.',
 '2. `step_started` [step_started] LLM completion started :: {"tool_count": 28, "iteration": 1}',
 '3. `tool_called` [tool] calling Tool called: ask_user :: args={"question": "What specific information would you like me to retrieve?", "options": [{"label": "Current Bitcoin price (USD)"}, {"label": "Current price of another cryptocurrency (e.g., Ethereum)"}, {"label": "Weather …',
 '4. `tool_completed` [tool] completed Tool completed: ask_user :: {"kind": "ask_user", "question": "What specific information would you like me to retrieve?", "context": "", "options": [{"label": "Current Bitcoin price (USD)"}, {"label": "Current price of another cryptocurrency …',
 '5. `interactive_request` [interactive] Interactive request from ask_user :: {"kind": "ask_user", "payload": {"interactive": true, "kind": "ask_user", "question": "What specific information would you like me to retrieve?", "context": ""

In [ ]:
# Collected event packets in one object.
events = [event.to_dict() for event in agent.stream(prompt)]
events

In [ ]:
# Final output plus stored events from a non-streaming run.
result = agent.run(prompt)
print(result.output)
result.events[:5]

In [ ]:
session = agent.chat_session(session_id='stream-demo')

for packet in session.stream_packets(
    'Use tools if useful and return a compact answer.',
    transport='websocket',
):
    print(packet)

In [ ]:
session = agent.chat_session(session_id='stream-demo-sse')

for packet in session.stream_packets(
    'Use tools if useful and return a compact answer.',
    transport='sse',
):
    print(packet)